# XGBoost: gradient-boosted trees

## Plan

1. **Why boosting.** The Decision Tree fits the training data almost perfectly, and its train−val gap is about 0. Its val−test gap, however, is about 0.21 macro-F1, and the main reason is that it misses novel attack types. The Random Forest averages many deep trees, which reduces variance. Gradient boosting works differently: it adds shallow trees one after another, and each new tree corrects the errors left by the previous ones. Shrinkage (`learning_rate`), row and column subsampling, and L1/L2 regularisation control complexity. This notebook asks whether boosting narrows the val−test gap.
2. **Protocol identical to LR / DT / RF.** Everything runs through `src/experiment.py`:
   - the same seed (42);
   - the same `train_sub`/`val` indices (`results/splits/`) and the same stratified 5-fold CV on `train_sub`;
   - Optuna TPE with the same trial budget, maximising CV macro-F1;
   - the threshold chosen on `val` (max macro-F1), then a refit on all of KDDTrain+;
   - **KDDTest+ scored once**;
   - MLflow logging plus artifacts in `models/` and `scalars/`.
3. **Preprocessing.** The shared `tree` family from `src/features.py`:
   - `FeatureCleaner`;
   - `CorrelationFilter(0.98)`;
   - OHE with `min_frequency=50`;
   - no `log1p` or scaling, because tree splits are invariant to monotone transforms.
4. **Search space:**
   - `n_estimators` ∈ [100, 800] (step 50)
   - `learning_rate` ∈ [0.01, 0.3] (log)
   - `max_depth` ∈ [3, 10]
   - `min_child_weight` ∈ [1, 20] (log)
   - `subsample` ∈ [0.6, 1.0]
   - `colsample_bytree` ∈ [0.5, 1.0]
   - `gamma` ∈ [0, 5]
   - `reg_alpha` ∈ [1e-8, 10] (log)
   - `reg_lambda` ∈ [1e-3, 10] (log)
   - `scale_pos_weight` ∈ {1, neg/pos}

   Fixed: `tree_method="hist"`, `objective="binary:logistic"`, `random_state=42`.
5. **Diagnostics specific to boosting:**
   - learning curves (logloss and error against the boosting round);
   - learning-rate × number-of-trees sensitivity;
   - the depth train/val gap.
6. **Comparison.** XGBoost against the majority baseline, LR, DT and RF. The comparison uses saved metrics and predictions only, with no refits. It covers test metrics, the generalization-gap table, ROC/PR overlays, per-category and novel-attack recall, and McNemar tests.

> Set the environment variable `XGB_SMOKE=1` before starting the kernel for a quick smoke test. It runs 2 trials on a 5k subsample. Do not report smoke-test numbers.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src import experiment as ex, plots
from src.features import build_pipeline, build_preprocessor
from sklearn.metrics import f1_score, roc_curve, precision_recall_curve
from xgboost import XGBClassifier, __version__ as xgb_version

rng = ex.init()
SEED, CFG = ex.SEED, ex.CFG
sns.set_theme(style="whitegrid")
MODEL = "xgboost"
PREP = dict(corr_threshold=CFG["features"]["corr_threshold"], min_frequency=CFG["features"]["min_frequency"])
SMOKE = os.environ.get("XGB_SMOKE") == "1"

train, test, novel = ex.load_data()
idx_tr, idx_va = ex.get_splits(train)
X_all, y_all = ex.xy(train)
X_tr, y_tr, X_va, y_va = X_all.iloc[idx_tr], y_all[idx_tr], X_all.iloc[idx_va], y_all[idx_va]
N_TRIALS = 2 if SMOKE else ex.n_trials()
if SMOKE:  # stratified 5k subsample for tuning only (smoke test)
    sub = pd.Series(y_tr).groupby(y_tr, group_keys=False).sample(n=2500, random_state=SEED).index.to_numpy()
    X_tune, y_tune = X_tr.iloc[sub], y_tr[sub]
else:
    X_tune, y_tune = X_tr, y_tr
SPW = float((y_tr == 0).sum() / (y_tr == 1).sum())  # neg/pos on train_sub
run_id = ex.start_run(MODEL, {"family": "tree", "log_transform": False, "smoke": SMOKE, **PREP})
print(f"xgboost {xgb_version} | MLflow run {run_id} | trials {N_TRIALS} | tuning rows {len(y_tune)} | neg/pos {SPW:.3f}"
      + ("  ** SMOKE TEST **" if SMOKE else ""))

## 1. Optuna search (5-fold CV macro-F1 on `train_sub`)

In [ ]:
XGB_FIXED = dict(objective="binary:logistic", tree_method="hist", eval_metric="logloss", importance_type="gain",
                 n_jobs=-1, random_state=SEED)

def make_estimator(p, **over):
    return XGBClassifier(**{**XGB_FIXED, **p, **over})

def make_pipeline(p):
    return build_pipeline(make_estimator(p), "tree", **PREP)

def suggest(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, round(SPW, 4)])}

t0 = time.perf_counter()
study = ex.tune(MODEL, run_id, make_pipeline, suggest, X_tune, y_tune, trials=N_TRIALS, cv_jobs=1)  # XGB uses all cores
tuning_seconds = time.perf_counter() - t0
best = study.best_params
print(f"best CV macro-F1 = {study.best_value:.4f} ({tuning_seconds/60:.1f} min)")
pd.Series(best, name="best").to_frame()

In [ ]:
import optuna
fig = plots.optuna_overview(study, "learning_rate", hue="max_depth"); ex.log_figure(run_id, fig, "optuna", MODEL); plt.show()
if len(study.trials) >= 5:
    pimp = pd.Series(optuna.importance.get_param_importances(study, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED)))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(pimp.index[::-1], pimp.values[::-1], color="#DD8452"); ax.set_title("Optuna hyperparameter importance (fANOVA)")
    fig.tight_layout(); ex.log_figure(run_id, fig, "optuna_param_importance", MODEL); plt.show()

## 2. Train / validation diagnostics, threshold and the single test evaluation

In [ ]:
res = ex.fit_and_evaluate(MODEL, run_id, make_pipeline, best, train, test, idx_tr, idx_va, novel,
                          extra={"tuning": {"best_cv_f1_macro": study.best_value, "n_trials": len(study.trials),
                                            "tuning_seconds": tuning_seconds, "smoke": SMOKE}})
pd.DataFrame({k: res["metrics"][k] for k in ["train", "val", "test", "test_at_0.5"]}).T[
    ["threshold", "accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "fpr", "fnr"]].round(4)

In [ ]:
fig = plots.metric_bars(res["metrics"], "XGBoost: train vs validation vs test"); ex.log_figure(run_id, fig, "metric_bars", MODEL); plt.show()
fig = plots.confusion_and_curves(res["y_va"], res["p_va"], res["y_te"], res["p_te"], res["threshold"], "XGB")
ex.log_figure(run_id, fig, "confusion_roc_pr_threshold", MODEL); plt.show()

## 3. Boosting diagnostics (fit on `train_sub`, scored on `train_sub` and `val`)

The preprocessor is fit once on `train_sub`, and each diagnostic fit reuses the tuned parameters with one setting changed. These fits never see the test set.

In [ ]:
prep = build_preprocessor("tree", **PREP).fit(X_tr)
Ztr, Zva = prep.transform(X_tr), prep.transform(X_va)

# 3a) learning curves: logloss / error per boosting round
lc = make_estimator(best, eval_metric=["logloss", "error"]).fit(Ztr, y_tr, eval_set=[(Ztr, y_tr), (Zva, y_va)], verbose=False)
ev = lc.evals_result()
rounds = np.arange(1, len(ev["validation_0"]["logloss"]) + 1)
best_round = int(np.argmin(ev["validation_1"]["logloss"])) + 1
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
for ax, metric in zip(axes, ["logloss", "error"]):
    ax.plot(rounds, ev["validation_0"][metric], label="train_sub"); ax.plot(rounds, ev["validation_1"][metric], label="val")
    ax.axvline(best_round, c="#8C8C8C", ls="--", lw=1, label=f"min val logloss @ {best_round}")
    ax.set(xlabel="boosting round", ylabel=metric, yscale="log", title=f"Learning curve: {metric}"); ax.legend()
fig.tight_layout(); ex.log_figure(run_id, fig, "learning_curves", MODEL); plt.show()
print(f"tuned n_estimators = {best['n_estimators']} | val logloss minimum at round {best_round}")

In [ ]:
def tv(**over):
    m = make_estimator(best, **over).fit(Ztr, y_tr)
    return f1_score(y_tr, m.predict(Ztr), average="macro"), f1_score(y_va, m.predict(Zva), average="macro")

# 3b) learning rate x number of trees (val macro-F1 at threshold 0.5)
grid = pd.DataFrame({(lr, n): tv(learning_rate=lr, n_estimators=n)[1] for lr in [0.03, 0.1, 0.3] for n in [100, 400]},
                    index=["val_f1_macro"]).T
grid.index.names = ["learning_rate", "n_estimators"]
# 3c) max_depth: training vs validation gap
depths = list(range(2, 13))
dcurve = pd.DataFrame([tv(max_depth=d) for d in depths], index=depths, columns=["train", "val"])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
sns.heatmap(grid["val_f1_macro"].unstack(), annot=True, fmt=".4f", cmap="Blues", ax=axes[0])
axes[0].set_title("val macro-F1: learning_rate x n_estimators")
axes[1].plot(dcurve.index, dcurve["train"], "o-", label="train_sub"); axes[1].plot(dcurve.index, dcurve["val"], "s-", label="val")
axes[1].axvline(best["max_depth"], c="#8C8C8C", ls="--", lw=1, label="tuned depth")
axes[1].set(xlabel="max_depth", ylabel="macro-F1", title="Depth: training vs validation"); axes[1].legend()
fig.tight_layout(); ex.log_figure(run_id, fig, "sensitivity", MODEL); plt.show()
dcurve.assign(gap=dcurve["train"] - dcurve["val"]).round(4).T

## 4. Interpretability: gain importance vs permutation importance

In [ ]:
from sklearn.inspection import permutation_importance
final = res["final"]
names = np.array([n.split("__", 1)[1] for n in final[:-1].get_feature_names_out()])
gain = pd.Series(final[-1].feature_importances_, index=names).sort_values(ascending=False)  # normalised gain
gain.to_csv(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_feature_importance.csv")
pi = permutation_importance(res["dev"], X_va, y_va, scoring="f1_macro", n_repeats=5, random_state=SEED, n_jobs=1)
perm = pd.Series(pi.importances_mean, index=X_va.columns).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
t = gain.head(20); axes[0].barh(t.index[::-1], t.values[::-1], color="#DD8452"); axes[0].set_title("Gain importance (final model, top 20)")
t = perm.head(20); axes[1].barh(t.index[::-1], t.values[::-1], color="#8172B2"); axes[1].set_title("Permutation importance on val (raw features, Δ macro-F1)")
fig.tight_layout(); ex.log_figure(run_id, fig, "feature_importance", MODEL); plt.show()

In [ ]:
# top-15 importance side by side with the other tree models (each normalised to sum 1)
tun = PROJECT_ROOT / "results" / "tuning"
imps = {"XGB gain": gain}
for label, f in [("DT impurity", "decision_tree_feature_importance.csv"), ("RF impurity", "random_forest_optuna_feature_importance.csv")]:
    if (tun / f).exists():
        imps[label] = pd.read_csv(tun / f, index_col=0).iloc[:, 0]
    else:
        print(f"missing {f}: run that notebook first")
imp_df = pd.DataFrame(imps).fillna(0)
imp_df = imp_df / imp_df.sum()
top = imp_df.loc[imp_df.max(axis=1).sort_values(ascending=False).index[:15]]
fig, ax = plt.subplots(figsize=(10, 6)); top[::-1].plot(kind="barh", ax=ax); ax.set_title("Top-15 features across tree models")
fig.tight_layout(); ex.log_figure(run_id, fig, "importance_vs_trees", MODEL); plt.show()

## 5. Error analysis

In [ ]:
fig = plots.error_analysis(res["preds"], test, "XGB"); ex.log_figure(run_id, fig, "error_analysis", MODEL); plt.show()
fig = plots.missed_attack_types(res["preds"]); ex.log_figure(run_id, fig, "missed_attack_types", MODEL); plt.show()

## 6. Comparison against the other models

Only saved outputs are used here: `results/metrics/*.json` and `results/predictions/*_test_predictions.csv`. Every model shares the same splits, folds, threshold rule and test set. Any model whose outputs are missing is skipped with a warning.

In [ ]:
MODELS = {"majority_baseline": "Majority baseline", "logistic_regression": "Logistic Regression",
          "decision_tree": "Decision Tree", "random_forest_optuna": "Random Forest", "xgboost": "XGBoost"}
COLORS = dict(zip(MODELS.values(), ["#8C8C8C", "#4C72B0", "#55A868", "#C44E52", "#DD8452"]))
M, P = {}, {}
for k, name in MODELS.items():
    mp = PROJECT_ROOT / "results" / "metrics" / f"{k}.json"
    pp = PROJECT_ROOT / "results" / "predictions" / f"{k}_test_predictions.csv"
    if mp.exists() and pp.exists():
        M[name], P[name] = json.loads(mp.read_text()), pd.read_csv(pp)
    else:
        print(f"WARNING: {k} outputs missing, skipped (run its notebook first)")
KEYS = ["accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "fpr", "fnr"]
table = pd.DataFrame({n: {**{k: m["metrics"]["test"][k] for k in KEYS},
                          "recall_novel": m.get("slices", {}).get("recall_novel", np.nan),
                          "threshold": m.get("threshold", 0.5),
                          "fit_s": m.get("timing", {}).get("final_fit_seconds", np.nan),
                          "infer_us_per_row": m.get("timing", {}).get("inference_us_per_row", np.nan)}
                      for n, m in M.items()}).T
table.round(4).style.highlight_max(subset=["accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "recall_novel"],
                                   color="#d4edda").highlight_min(subset=["fpr", "fnr"], color="#d4edda").format(precision=4)

### 6a. Generalization gap

- **train − val** measures overfitting within the training distribution.
- **val − test** measures the drop caused by the shift from KDDTrain+ to KDDTest+: 17 novel attack types, and 57% attacks in test vs 47% in train.

For FPR and FNR, a negative gap means the test set is worse.

In [ ]:
rows = []
for n, m in M.items():
    if "train" not in m["metrics"] or "val" not in m["metrics"]:
        continue
    for k in KEYS:
        tr, va, te = (m["metrics"][s][k] for s in ["train", "val", "test"])
        rows.append({"model": n, "metric": k, "train": tr, "val": va, "test": te, "train-val": tr - va, "val-test": va - te})
gap = pd.DataFrame(rows)
gap.to_csv(PROJECT_ROOT / "results" / "tuning" / "generalization_gap_all_models.csv", index=False)
display(gap.pivot_table(index="metric", columns="model", values=["train-val", "val-test"]).round(4))

g = gap[gap.metric == "f1_macro"].set_index("model")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
for n, r in g.iterrows():
    axes[0].plot(["train", "val", "test"], r[["train", "val", "test"]], "o-", c=COLORS[n], label=n)
axes[0].set(ylabel="macro-F1", title="macro-F1 across splits"); axes[0].legend()
g[["train-val", "val-test"]].plot(kind="bar", ax=axes[1], rot=15, color=["#4C72B0", "#C44E52"])
axes[1].set(ylabel="macro-F1 gap", title="Generalization gap (macro-F1)")
for c in axes[1].containers: axes[1].bar_label(c, fmt="%.3f", fontsize=8)
fig.tight_layout(); ex.log_figure(run_id, fig, "generalization_gap", MODEL); plt.show()

### 6b. Test metrics, ROC / PR overlays and recall by attack category

In [ ]:
show = ["f1_macro", "recall", "precision", "fpr", "roc_auc", "pr_auc", "recall_novel"]
long = table[show].reset_index(names="model").melt(id_vars="model", var_name="metric")
fig, ax = plt.subplots(figsize=(15, 4.5))
sns.barplot(data=long, x="metric", y="value", hue="model", palette=COLORS, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=6)
ax.set_title("KDDTest+ metrics (threshold chosen on val)"); fig.tight_layout(); ex.log_figure(run_id, fig, "test_metric_bars", MODEL); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for n, p in P.items():
    if n == "Majority baseline":
        continue
    fpr_, tpr_, _ = roc_curve(p.y_true, p.y_prob); axes[0].plot(fpr_, tpr_, c=COLORS[n], label=f"{n} (AUC {M[n]['metrics']['test']['roc_auc']:.3f})")
    pr_, rc_, _ = precision_recall_curve(p.y_true, p.y_prob); axes[1].plot(rc_, pr_, c=COLORS[n], label=f"{n} (AP {M[n]['metrics']['test']['pr_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], "k:", lw=1); axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC (test)"); axes[0].legend(fontsize=8)
axes[1].axhline(P["XGBoost"].y_true.mean(), c="k", ls=":", lw=1); axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall (test)"); axes[1].legend(fontsize=8)
cat = pd.DataFrame({n: p[p.y_true == 1].groupby("attack_category")["y_pred"].mean() for n, p in P.items() if n != "Majority baseline"})
nov = pd.DataFrame({n: p[p.y_true == 1].groupby("is_novel")["y_pred"].mean().rename({0: "known types", 1: "novel types"})
                    for n, p in P.items() if n != "Majority baseline"})
pd.concat([cat, nov]).plot(kind="bar", ax=axes[2], rot=30, color=[COLORS[c] for c in cat.columns])
axes[2].set(ylabel="attack recall", title="Recall by attack category / novelty (test)"); axes[2].legend(fontsize=8)
fig.tight_layout(); ex.log_figure(run_id, fig, "roc_pr_category", MODEL); plt.show()
pd.concat([cat, nov]).round(4)

### 6c. McNemar tests (XGBoost vs each model, paired test predictions)

This is an exact McNemar test. Let b = the number of rows where XGB is right and the other model is wrong, and c = the number where XGB is wrong and the other model is right. Under H0, b ~ Binomial(b + c, 0.5). P-values are Holm-corrected across the comparisons.

In [ ]:
from scipy.stats import binomtest
xg = P["XGBoost"]
ok_x = (xg.y_pred == xg.y_true).to_numpy()
mc = []
for n, p in P.items():
    if n == "XGBoost":
        continue
    assert (p.row_id.to_numpy() == xg.row_id.to_numpy()).all(), f"row order differs for {n}"
    ok_o = (p.y_pred == p.y_true).to_numpy()
    b, c = int((ok_x & ~ok_o).sum()), int((~ok_x & ok_o).sum())
    mc.append({"vs": n, "xgb_right_other_wrong": b, "xgb_wrong_other_right": c,
               "p_value": binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0})
mc = pd.DataFrame(mc).sort_values("p_value").reset_index(drop=True)
k = len(mc)
mc["p_holm"] = np.maximum.accumulate(np.minimum(1, (k - np.arange(k)) * mc["p_value"]))
mc["significant_0.05"] = mc["p_holm"] < 0.05
mc

## 7. Summary

In [ ]:
m, s = res["metrics"], res["slices"]
others = table.drop(index=["XGBoost", "Majority baseline"], errors="ignore")
best_other = others["f1_macro"].idxmax() if len(others) else None
lines = [
    f"- **Best parameters** (CV macro-F1 {study.best_value:.4f}, {len(study.trials)} trials): `{best}`",
    f"- **Threshold** (val): {res['threshold']:.2f}",
    f"- **macro-F1** train / val / test: {m['train']['f1_macro']:.4f} / {m['val']['f1_macro']:.4f} / {m['test']['f1_macro']:.4f}"
    f" (gaps: train−val {m['train']['f1_macro'] - m['val']['f1_macro']:+.4f}, val−test {m['val']['f1_macro'] - m['test']['f1_macro']:+.4f})",
    f"- **Test** recall {m['test']['recall']:.4f}, FPR {m['test']['fpr']:.4f}, ROC-AUC {m['test']['roc_auc']:.4f}, PR-AUC {m['test']['pr_auc']:.4f}",
    f"- **Novel-attack recall** {s['recall_novel']:.4f} vs known {s['recall_known']:.4f}",
    f"- **Complexity** {res['payload']['complexity']} | ONNX: {res['payload']['artifacts'].get('onnx_status')}",
]
if best_other:
    d = table.loc["XGBoost", "f1_macro"] - table.loc[best_other, "f1_macro"]
    lines.append(f"- **vs best other model** ({best_other}): test macro-F1 {d:+.4f}")
if SMOKE:
    lines.insert(0, "> **SMOKE TEST: numbers are not valid results.**")
display(Markdown("\n".join(lines)))